### [Trading Strategies: Double Moving Average Crossover](https://medium.com/@brianhulela/trading-strategies-double-moving-average-crossover-36a3e9588510)

The Double Moving Average Crossover is one of the simplest yet powerful trading strategies used by both beginners and seasoned traders. By using two different moving averages, this strategy identifies potential buy and sell signals when the faster moving average crosses over or under a slower one.

The Double Moving Average Crossover strategy relies on two exponential moving averages (EMAs):

- **Short-term Moving Average (fast)**: This moving average responds quickly to recent price changes.
- **Long-term Moving Average (slow)**: This one reacts more slowly to price changes, capturing long-term trends.

##### Key Signals:
- **Buy Signal**: When the short-term EMA crosses above the long-term EMA, indicating an upward trend.
- **Sell Signal**: When the short-term EMA crosses below the long-term EMA, indicating a downward trend.

In [ ]:
!pip install -Uq pandas numpy matplotlib yfinance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from tabulate import tabulate
from datetime import datetime

In [ ]:
# Define the stock symbol and the date range for our data
stock_symbol = 'TSLA'
start_date = '2024-01-01'
end_date = datetime.today().strftime('%Y-%m-%d')  # Sets end date to today's date
print(f"Double Moving Average Crossover Trading for: {stock_symbol}\nStart Date: {start_date}\nEnd Date: {end_date}")

In [ ]:
df = yf.download(stock_symbol, start=start_date, end=end_date)

# Select the desired columns (first level of MultiIndex)
df.columns = df.columns.get_level_values(0)

# Keep only the columns you are interested in
df = df[['Open', 'Close', 'Volume', 'Low', 'High']]

# If the index already contains the dates, rename the index
df.index.name = 'Date'  # Ensure the index is named "Date"
    
# Resetting the index if necessary
df.reset_index(inplace=True)

# Ensure that the index is of type datetime
df['Date'] = pd.to_datetime(df['Date'])

# Set the 'Date' column as the index again (in case it's reset)
df.set_index('Date', inplace=True)

df.head()

In [ ]:
# Plot the closing price
plt.figure(figsize=(12, 6))

plt.plot(df['Close'], label='Closing Price')

# Add title, labels, and legend
plt.title(f'{stock_symbol} Closing Price')
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.legend()

# Save the plot in 300dpi
plt.savefig(f'{stock_symbol}_stock_chart.png', dpi=300)

# Show the plot
plt.show()

In [ ]:
# Calculate the short and long EMAs
SHORT_WINDOW = 10
LONG_WINDOW = 30
df['SHORT_WINDOW'] = df['Close'].ewm(span=SHORT_WINDOW, adjust=False).mean()
df['LONG_WINDOW'] = df['Close'].ewm(span=LONG_WINDOW, adjust=False).mean()

In [ ]:
# Plot the Close Price and EMAs
plt.figure(figsize=(14, 7))

# Plot Close Price
plt.plot(df.index, df['Close'], label='Close Price', linewidth=2)

# Plot Short-term EMA
plt.plot(df.index, df['SHORT_WINDOW'], label=f'{SHORT_WINDOW}-day EMA', linestyle='--')

# Plot Long-term EMA
plt.plot(df.index, df['LONG_WINDOW'], label=f'{LONG_WINDOW}-day EMA', linestyle='--')

# Labels and title
plt.xlabel('Date')
plt.ylabel('Price ($)')
plt.title(f'{stock_symbol} Close Price with Short and Long EMAs')
plt.legend()

# Save plot in 300dpi
plt.savefig(f'{stock_symbol}_moving_averages.png', dpi=300)

plt.show()

In [ ]:
# Generate buy and sell signals in a single column
df['Signal'] = np.where(
    (df['SHORT_WINDOW'] > df['LONG_WINDOW']) & (df['SHORT_WINDOW'].shift(1) <= df['LONG_WINDOW'].shift(1)), 1,  # Buy Signal
    np.where(
        (df['SHORT_WINDOW'] < df['LONG_WINDOW']) & (df['SHORT_WINDOW'].shift(1) >= df['LONG_WINDOW'].shift(1)), -1,  # Sell Signal
        0  # No Signal
    )
)

In [ ]:
# Define the fee calculation function
def calculate_fee(amount: float) -> float:
    """Calculate the brokerage fee based on transaction amount."""
    fee = amount * 0.0025  # 0.25% of the transaction
    return max(fee, 0.01)   # Minimum fee of $0.01

In [ ]:
# Starting capital
initial_cash = 100  # Example initial capital
cash = initial_cash
shares = 0
df['Portfolio Value'] = cash

# List to store transaction details for tabulation
transaction_details = []

for i, row in df.iterrows():
    if row['Signal'] == 1 and cash > 0:  # Buy condition
        # Calculate the fee for the buy transaction
        fee = calculate_fee(cash)
        
        # Calculate how many shares can be bought after deducting the fee
        shares_bought = (cash - fee) / row['Close']
        
        # Deduct the cash and fee for the purchase
        cash -= shares_bought * row['Close'] + fee
        
        # Add the bought shares to the portfolio
        shares += shares_bought
        
        # Record transaction details
        transaction_details.append([row.name, 'Buy', round(row['Close'], 2), round(fee, 2), round(cash + (shares * row['Close']), 2)])

    elif row['Signal'] == -1 and shares > 0:  # Sell condition
        # Calculate how much cash is earned from selling shares
        total_sale_amount = shares * row['Close']
        
        # Calculate the fee for the sell transaction
        fee = calculate_fee(total_sale_amount)
        
        # Cash earned after the fee is deducted
        cash_earned = total_sale_amount - fee
        
        # Update the cash balance after selling shares
        cash += cash_earned
        
        # All shares are sold
        shares = 0
        
        # Record transaction details
        transaction_details.append([row.name, 'Sell', round(row['Close'], 2), round(fee, 2), round(cash, 2)])

    # Update portfolio value
    df.at[i, 'Portfolio Value'] = cash + (shares * row['Close'])

# Summarize results using tabulate
print(tabulate(transaction_details, headers=["Date", "Action", "Price ($)", "Fee ($)", "Portfolio Value ($)"], tablefmt="pretty"))

# Final performance
final_value = cash + (shares * df.iloc[-1]['Close'])
profit = final_value - initial_cash
print(f"\nFinal Portfolio Value: ${final_value:.2f}")
print(f"Total Profit/Loss: ${profit:.2f}")